# 05 — M2 Sexual Harassment Corpus V2

**Goal:** Build the sexual harassment corpus for M2 four-class router.

**V2 Changes:**
1. `normalize_text()` on ALL text
2. **CASH RE-ADDED** with direct-attack filter (was excluded from M2 V1)
3. **TRAC-3 sexual REMOVED** (not English)
4. Cap ~7,000 rows for class balance

**Sources:** EDOS (derog + animosity), SBIC (sexual), CASH (direct-attack), AEGIS, ConvAbuse, InViS

In [ ]:
# ============================================================
# CELL 1 — Setup + Shared Normalizer
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import re
import hashlib

BASE      = Path("/content/drive/MyDrive/AEGIS_ML")
RAW       = BASE / "data" / "raw"
THREAT    = RAW / "threat_gap"
PROCESSED = BASE / "data" / "processed" / "v5"
PROCESSED.mkdir(parents=True, exist_ok=True)

def normalize_text(text: str) -> str:
    """Canonical text normalization for AEGIS pipeline."""
    text = str(text).strip()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\brt\s+', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'<user>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else None

def text_hash(t):
    return hashlib.sha256(t.strip().encode('utf-8')).hexdigest()

print("✅ Setup complete")

In [ ]:
# ============================================================
# CELL 2 — Load All Sexual Harassment Sources
# ============================================================

frames = []

print("="*70)
print("SEXUAL HARASSMENT CORPUS V2 — LOADING SOURCES")
print("="*70)

# ── 1. EDOS — derogation + animosity ─────────────────────────
print("\n[1] EDOS (derogation + animosity)")
df = pd.read_csv(RAW / 'edos.csv')
target_cats = ['2. derogation', '3. animosity']
df = df[df['label_category'].isin(target_cats)].copy()
df['text'] = df['text'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'sexual_harassment'
frames.append(df[['text', 'label']].copy())
print(f"  ✅ {len(df):,} rows")

# ── 2. SBIC — sexual content ─────────────────────────────────
print("\n[2] SBIC (sexYN >= 0.5 & offensiveYN >= 0.5)")
df = pd.read_csv(RAW / 'sbic_social_bias_frames.csv', low_memory=False)
df['sexYN'] = pd.to_numeric(df['sexYN'], errors='coerce').fillna(0)
df['offensiveYN'] = pd.to_numeric(df['offensiveYN'], errors='coerce').fillna(0)
df = df[(df['sexYN'] >= 0.5) & (df['offensiveYN'] >= 0.5)].copy()
# Deduplicate by post text
df = df.drop_duplicates(subset='post').copy()
df['text'] = df['post'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'sexual_harassment'
frames.append(df[['text', 'label']].copy())
print(f"  ✅ {len(df):,} rows")

# ── 3. CASH (RE-ADDED — direct-attack filter) ────────────────
print("\n[3] CASH (direct-attack patterns only)")
df = pd.read_csv(RAW / 'cash_sexual_harassment_en.csv')
# Filter to keep only direct-attack patterns, not victim narratives
DIRECT_PATTERNS = r'\b(you|your|ur|send|show|give|let me|suck|fuck|bitch|slut|whore|tits|ass|pussy|dick|cock|nude|naked|sexy|hot|wanna|come|baby|daddy|honey|sugar|darling|sweetie|dm|pics|snap)\b'
df['text_lower'] = df['text'].str.lower()
df = df[df['text_lower'].str.contains(DIRECT_PATTERNS, regex=True, na=False)].copy()
df = df.drop(columns='text_lower')
df['text'] = df['text'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'sexual_harassment'
frames.append(df[['text', 'label']].copy())
print(f"  ✅ {len(df):,} rows (filtered from 18,096 raw)")

# ── 4. AEGIS — sexual labels ─────────────────────────────────
print("\n[4] AEGIS Safety (sexual labels)")
try:
    df = pd.read_csv(RAW / 'aegis_safety_corpus.csv')
    df = df[df['text_type'] == 'user_message'].copy()
    label_cols = ['labels_0','labels_1','labels_2','labels_3','labels_4']
    sex_labels = ['sexual content']
    
    def is_sexual(row):
        votes = [str(row[c]).strip().lower() for c in label_cols if pd.notna(row[c])]
        if not votes: return False
        sex_votes = sum(1 for v in votes if v in sex_labels)
        return sex_votes > len(votes)/2
    
    df_sex = df[df.apply(is_sexual, axis=1)].copy()
    df_sex['text'] = df_sex['text'].apply(normalize_text)
    df_sex = df_sex[df_sex['text'].notna() & (df_sex['text'].str.len() > 0)]
    df_sex['label'] = 'sexual_harassment'
    frames.append(df_sex[['text', 'label']].copy())
    print(f"  ✅ {len(df_sex):,} rows")
except Exception as e:
    print(f"  ⚠️ AEGIS extraction failed: {e}")

# ── 5. ConvAbuse sexual harassment ───────────────────────────
print("\n[5] ConvAbuse sexual harassment")
try:
    df = pd.read_csv(THREAT / 'convabuse_sexual_harassment.csv')
    df['text'] = df['text'].apply(normalize_text)
    df = df[df['text'].notna() & (df['text'].str.len() > 0)]
    df['label'] = 'sexual_harassment'
    frames.append(df[['text', 'label']].copy())
    print(f"  ✅ {len(df):,} rows")
except: print("  ⚠️ File not found")

# ══════════════════════════════════════════════════════════════
# ❌ REMOVED: TRAC-3 sexual harassment (Bengali/Hindi)
# ══════════════════════════════════════════════════════════════
print("\n❌ SKIPPED: TRAC-3 sexual harassment (not English)")

# ── COMBINE ──────────────────────────────────────────────────
print("\n" + "="*70)
print("COMBINING SEXUAL HARASSMENT SOURCES")
print("="*70)

df_sh_all = pd.concat(frames, ignore_index=True)

# Dedup
before = len(df_sh_all)
df_sh_all['text_hash'] = df_sh_all['text'].apply(text_hash)
df_sh_all = df_sh_all.drop_duplicates(subset='text_hash', keep='first').copy()
df_sh_all = df_sh_all.drop(columns='text_hash')
print(f"  Dedup: {before:,} → {len(df_sh_all):,} ({before-len(df_sh_all):,} removed)")

# Filter min words
df_sh_all = df_sh_all[df_sh_all['text'].str.split().str.len() >= 3].copy()

# Cap at 7,000 for class balance
CAP = 7_000
if len(df_sh_all) > CAP:
    df_sh_all = df_sh_all.sample(n=CAP, random_state=42)
    print(f"  Capped to {CAP:,} for M2 class balance")

print(f"\n  FINAL SEXUAL HARASSMENT CORPUS: {len(df_sh_all):,} rows")

# Spot check
print("\n📌 Random sample (10 examples):")
for _, row in df_sh_all.sample(min(10, len(df_sh_all)), random_state=42).iterrows():
    print(f"  • {row['text'][:100]}")

In [ ]:
# ============================================================
# CELL 3 — Save Sexual Harassment Corpus
# ============================================================

print("="*70)
print("SAVING SEXUAL HARASSMENT CORPUS V2")
print("="*70)

df_sh_all['language'] = 'en'
df_sh_all['source_dataset'] = 'sexual_harassment_v2'
df_sh_all['routing_method'] = 'manual'

out_path = PROCESSED / "m2_sexual_harassment_v2.csv"
df_sh_all.to_csv(out_path, index=False)

size_kb = out_path.stat().st_size / 1024
print(f"\n  ✅ Saved: {out_path.name} ({size_kb:.1f} KB)")
print(f"  ✅ Rows: {len(df_sh_all):,}")

# Validation
has_upper = df_sh_all['text'].str.contains(r'[A-Z]', regex=True).sum()
has_url   = df_sh_all['text'].str.contains(r'https?://', regex=True).sum()
print(f"\n  Validation:")
print(f"    Uppercase chars: {has_upper} (should be 0)")
print(f"    URLs remaining: {has_url} (should be 0)")
print(f"\n🎉 Sexual Harassment V2 complete!")